# TrainTrace: Viva and run analysis notebook

This notebook is an analysis stage for the TrainTrace pipeline. Select a run and inspect its actual per-epoch metrics, holdout evaluation, confusion matrix, configuration, and readable artifact files. Training jobs still use TrainTrace’s durable queue; the notebook reads the same SQLite and artifact outputs used by the web app.

Epoch metrics are available in SQLite and in an incrementally refreshed CSV after each completed epoch, including when a run is later stopped or fails. Holdout metrics are stored as JSON after evaluation. Checkpoints (`.pt`) contain model/optimizer tensors and are inspected by tensor name and shape below.

In [ ]:
from pathlib import Path
import os, sys, json, sqlite3, csv

# Locate the project even if Jupyter was opened from notebooks/.
CANDIDATES = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in CANDIDATES if (p / "backend" / "core" / "trainer.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the TrainTrace checkout.")
BACKEND = ROOT / "backend"
sys.path.insert(0, str(BACKEND))
from config import DB_PATH, ARTIFACTS_DIR, REGISTRY_DIR
print("Project:", ROOT)
print("Database:", DB_PATH)
print("Artifacts:", ARTIFACTS_DIR)

## 1. Read the project files

A notebook can open files that VS Code does not render as readable documents, including binary checkpoint files. Text files are shown with paths and line numbers; binary files should be inspected through their metadata and tensors rather than printed as raw bytes.

In [ ]:
def show_text(relative_path, start=1, end=100):
    path = ROOT / relative_path
    lines = path.read_text(encoding="utf-8").splitlines()
    print(f"--- {relative_path} ({len(lines)} lines) ---")
    for number, line in enumerate(lines[start-1:end], start):
        print(f"{number:4}: {line}")

show_text("README.md", 1, 75)

## 2. Understand the model and training loop

The model factory creates one of four architecture families. CustomCNN exposes the number of convolution blocks, starting channel width, classifier width, and dropout. This selected profile is stored in the immutable run configuration and checkpoint, so inference can rebuild the same network shape later. The three preset torchvision families retain their fixed backbone structure.

In [ ]:
show_text("backend/core/models.py", 1, 120)
show_text("backend/core/trainer.py", 1, 100)

## 3. Select a run and read its pipeline outputs

Run the next cell to list actual runs in the local database. Set `RUN_ID` to the candidate you want to review, then run the analysis cell. It reads epoch rows directly from SQLite, the readable per-run CSV (when available), and the holdout evaluation JSON. Failed and stopped runs still show their completed epoch metrics.

Metrics use percent for accuracy, seconds for epoch duration, and the recorded learning-rate units. This project currently writes CSV/JSON and SQLite for metrics; it does not generate `.pkl` metric files. The `.pt` files are model checkpoints, not metric tables.

In [ ]:
RUNS = []
if not DB_PATH.is_file():
    print('No database yet. Start the TrainTrace API, then rerun this cell.')
else:
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        tables = {row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        if 'runs' in tables:
            columns = {row[1] for row in conn.execute("PRAGMA table_info(runs)")}
            fields = ['id', 'approach_name', 'model_architecture', 'status', 'epochs', 'seed']
            if 'config_json' in columns:
                fields.append('config_json')
            RUNS = [dict(r) for r in conn.execute('SELECT ' + ', '.join(fields) + ' FROM runs ORDER BY rowid DESC')]
            for run in RUNS:
                config_json = run.pop('config_json', None)
                config = json.loads(config_json) if config_json else {}
                run['dataset_name'] = config.get('dataset_name', 'Unknown')
            if not RUNS:
                print('The database is ready, but no training runs have been recorded yet.')
        else:
            print('The database has no TrainTrace run table yet. Start the API once to initialize it.')
[(r['id'], r.get('approach_name'), r.get('dataset_name'), r.get('model_architecture'), r.get('status'), r.get('epochs'), r.get('seed')) for r in RUNS[:50]]

In [ ]:
# Choose a run ID from RUNS above. The default selects the newest run.
RUN_ID = RUNS[0]['id'] if RUNS else None
RUN_ID

In [ ]:
from IPython.display import display, HTML, FileLink
from html import escape

def inspect_run(run_id):
    if not run_id:
        print('No runs yet. Create a run in TrainTrace, then rerun the run-list cell.')
        return None, [], None
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute('SELECT * FROM runs WHERE id=?', (run_id,)).fetchone()
        if row is None:
            raise ValueError(f'Unknown run ID: {run_id}')
        metrics = [dict(r) for r in conn.execute('SELECT * FROM epoch_metrics WHERE run_id=? ORDER BY epoch, id', (run_id,))]
        latest_decision = conn.execute('SELECT status FROM decisions WHERE run_id=? ORDER BY id DESC LIMIT 1', (run_id,)).fetchone()
    run = dict(row)
    config = json.loads(run['config_json']) if run.get('config_json') else {}
    dataset = json.loads(run['dataset_json']) if run.get('dataset_json') else {}
    final_record = json.loads(run['final_evaluation_json']) if run.get('final_evaluation_json') else None
    final_visible = (run.get('final_evaluation_status') == 'COMPLETED' and latest_decision and latest_decision['status'] == 'SELECTED' and final_record and not final_record.get('legacy_preselection_test_exposure'))
    evaluation = final_record if final_visible else None
    print(f"Run {run_id} · {run['status']} · {dataset.get('name', config.get('dataset_name', 'Unknown dataset'))} · {run['model_architecture']} · seed {run['seed']}")
    display(HTML('<h4>Recorded configuration</h4><pre>' + escape(json.dumps(config, indent=2)) + '</pre>'))
    if metrics:
        columns = ['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 'learning_rate', 'epoch_time_sec']
        header = ''.join(f'<th>{escape(c)}</th>' for c in columns)
        rows = []
        for metric in metrics:
            cells = [escape(str(metric.get(column, ""))) for column in columns]
            rows.append("<tr>" + "".join(f"<td>{cell}</td>" for cell in cells) + "</tr>")
        rows = "".join(rows)
        display(HTML('<h4>Per-epoch metrics (accuracy in percent)</h4><div style="overflow-x:auto"><table><thead><tr>' + header + '</tr></thead><tbody>' + rows + '</tbody></table></div>'))
    else:
        print('No completed epoch metrics have been recorded yet.')
    csv_path = ARTIFACTS_DIR / 'runs' / run_id / 'metrics' / f'{run_id}_metrics.csv'
    if not csv_path.is_file():
        csv_path = ARTIFACTS_DIR / f'{run_id}_metrics.csv'
    if csv_path.is_file():
        print('Human-readable CSV artifact:')
        display(FileLink(str(csv_path)))
    else:
        print('Metrics CSV will appear after the first completed epoch.')
    if evaluation:
        display(HTML('<h4>Holdout evaluation (accuracy in percent)</h4><pre>' + escape(json.dumps(evaluation, indent=2)) + '</pre>'))
        matrix = evaluation.get('confusion_matrix')
        if matrix:
            print('Confusion matrix: rows=true class, columns=predicted class')
            labels = dataset.get('classes') or [str(i) for i in range(len(matrix))]
            header = '<tr><th>True / Predicted</th>' + ''.join(f'<th>{escape(name)}</th>' for name in labels) + '</tr>'
            body = ''.join('<tr><th>' + escape(labels[i]) + '</th>' + ''.join(f'<td>{v}</td>' for v in line) + '</tr>' for i, line in enumerate(matrix))
            display(HTML('<div style="max-height:32rem;overflow:auto"><table><caption>Confusion matrix; rows=true class, columns=predicted class</caption><thead>' + header + '</thead><tbody>' + body + '</tbody></table></div>'))
    else:
        print('Final holdout evidence is sealed or unavailable. It appears only after a selected candidate completes its reserved final evaluation.')
    return run, metrics, evaluation

selected_run, selected_metrics, selected_evaluation = inspect_run(RUN_ID)

## 4. Open a checkpoint safely

A `.pt` checkpoint is a binary serialization, not a text file. The following cell lists checkpoint artifacts and the next cell reads a selected checkpoint using PyTorch's restricted `weights_only=True` loader. Tensor names/shapes and the captured model profile help explain the trained network. Metric values are shown in the selected-run analysis above.

In [ ]:
import torch

checkpoint_files = sorted(ARTIFACTS_DIR.glob("runs/*/checkpoints/*_checkpoint.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
checkpoint_files += sorted(ARTIFACTS_DIR.glob("*_checkpoint.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
[(p.name, p.stat().st_size) for p in checkpoint_files[:20]]

In [ ]:
if checkpoint_files:
    checkpoint_path = checkpoint_files[0]  # Change this to inspect another listed artifact.
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    print("File:", checkpoint_path.name)
    print("Run:", checkpoint.get("run_id"))
    print("Epoch:", checkpoint.get("epoch"))
    print("Configuration:", json.dumps(checkpoint.get("config", {}), indent=2))
    print("State tensors:")
    for name, tensor in checkpoint["model_state_dict"].items():
        print(f"  {name:48} {tuple(tensor.shape)} {tensor.dtype}")
else:
    print("No checkpoints found yet. Queue a run in the web app first.")

## 5. Follow the complete project flow

1. Create an investigation with an objective and hypothesis.
2. In Training, choose CIFAR-10 or CIFAR-100, then choose the model family and run controls. For CustomCNN, set feature-block depth and width explicitly.
3. Queue a run and inspect live progress; the pipeline writes the readable metrics CSV after every completed epoch.
4. Open this notebook, select that run, and read its epoch table, holdout metrics, and confusion matrix; then compare candidates in the app.
5. Record limitations and alternatives before selecting a candidate.
6. Register the selected run as an immutable model version, then activate it.
7. Use Inference to predict with the active version; the response identifies the exact version and source run.

The notebook is a first-class analysis companion to the tracked training pipeline. The web workflow remains the authoritative path for launching tracked jobs and changing decisions.